In [14]:
import json
import pandas as pd
from pathlib import Path
import uuid

from dsp_interview_transcripts import PROJECT_DIR
from dsp_interview_transcripts.utils.data_cleaning import clean_data

# app-specific utils (but maybe these should go elsewhere?)
from dsp_interview_transcripts.app_rq.utils.llm_question_answering import build_question_prompt_dict, parse_rqs, run_batch_check_for_all_rqs, PROMPT_PATH, concat_batch_check_output
from dsp_interview_transcripts.app_rq.utils.llm_question_answering import convert_transcripts_df_to_dict
from dsp_interview_transcripts.app_rq.utils.dash_utils import get_or_create_output_dir
from dsp_interview_transcripts.app_rq.utils.llm_question_answering import run_batch_check

In [15]:
qualaf_data = pd.read_csv('s3://dsp-qualfml/bus/raw/qualaf_test_conversations.csv')

In [16]:
qualaf_data

,uuid,created_at,timestamp,conversation,role,is_hidden,is_pending,text,audio,audio_url,transcript,image,image_url,caption
0,018fb90c-7029-2da4-a7a7-56036549daab,2024-05-27 08:55:51.210789+01:00,2024-05-27 08:55:49+01:00,018fb90c-7014-6318-28bb-86a49bf37d97,USER,False,False,Hello!,NaN,NaN,NaN,NaN,NaN,NaN
1,018fb90c-76fa-9b31-9e99-d537711d8d34,2024-05-27 08:55:52.956853+01:00,2024-05-27 08:55:52.953982+01:00,018fb90c-7014-6318-28bb-86a49bf37d97,BOT,False,False,"Hello! I am Alex, BIT and Nestaâ€™s digital in...",NaN,NaN,NaN,NaN,NaN,NaN
2,018fb90c-b750-294f-0117-2835bee86db8,2024-05-27 08:56:09.427151+01:00,2024-05-27 08:56:08+01:00,018fb90c-7014-6318-28bb-86a49bf37d97,USER,False,False,I consent!,NaN,NaN,NaN,NaN,NaN,NaN
3,018fb90c-dca1-8309-c72d-d70ded384c97,2024-05-27 08:56:18.982822+01:00,2024-05-27 08:56:18.977227+01:00,018fb90c-7014-6318-28bb-86a49bf37d97,BOT,False,False,Great! Your unique participant ID is: wOa926.,NaN,NaN,NaN,NaN,NaN,NaN
4,018fb90c-e2cc-1003-387c-ccaadc9b46a5,2024-05-27 08:56:20.561056+01:00,2024-05-27 08:56:20.556779+01:00,018fb90c-7014-6318-28bb-86a49bf37d97,BOT,False,False,Please make sure to put this ID in your post-i...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,018fe2f9-855c-9f6b-77d6-0e7a66f59500,2024-06-04 12:19:14.525078+01:00,2024-06-04 12:19:13+01:00,018fe2d4-8daa-33c2-3e17-5bfdcf79e54a,USER,False,False,No as it would have to make sense to do so for...,NaN,NaN,NaN,NaN,NaN,NaN
347,018fe2f9-bf40-f49b-d528-a11b308aab11,2024-06-04 12:19:29.347863+01:00,2024-06-04 12:19:29.344720+01:00,018fe2d4-8daa-33c2-3e17-5bfdcf79e54a,BOT,False,False,Thank you for participating! There are no furt...,NaN,NaN,NaN,NaN,NaN,NaN
348,018fe2fa-1b80-50e5-3400-9afc08076369,2024-06-04 12:19:52.960492+01:00,2024-06-04 12:19:51+01:00,018fe2d4-8daa-33c2-3e17-5bfdcf79e54a,USER,False,False,Thanks.,NaN,NaN,NaN,NaN,NaN,NaN
349,018fe2fa-56e3-7d34-c1d7-a73b8577ad76,2024-06-04 12:20:08.166228+01:00,2024-06-04 12:20:08.163373+01:00,018fe2d4-8daa-33c2-3e17-5bfdcf79e54a,BOT,False,False,Thank you for taking part in this interview! Y...,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Step 0: data cleaning (happens in the 'Upload' tab)
cleaned_df = clean_data(qualaf_data, 'text')

In [18]:
rq_text = """
Are participants positive about the interview experience?
Do participants believe the eligibility requirements are fair?
"""

In [19]:
output_dir = get_or_create_output_dir('notebook_test', test_mode=False)

In [ ]:
# Step 1: batch_check
output_paths, rq_dict = run_batch_check_for_all_rqs(
    rq_text=rq_text,
    cleaned_df=cleaned_df,
    output_dir=output_dir,
    conv_col='conversation',
    role_col='role',
    uuid_col='uuid'
)

2025-05-01 11:56:17,278 - root - INFO - Using Azure OpenAI
2025-05-01 11:56:17,294 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-05-01 11:56:17,305 - root - INFO - Using Azure OpenAI
2025-05-01 11:56:17,316 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client


2025-05-01 11:56:17,328 - root - INFO - Processing batch 1/1
2025-05-01 11:56:17,329 - root - INFO - Processing batch 1/1


In [21]:
df_output = concat_batch_check_output(rq_dict, output_paths)

,answer,explanation,text,identifier,id,timestamp,model,temperature,rq
0,yes,The participant expresses positive sentiments ...,"[I think this is an important topic, and home ...","[018fb938-7562-538a-78bb-5e3dd105e831, 018fb93...",018fb90c-7014-6318-28bb-86a49bf37d97,2025-05-01 10:56:17.330143+00:00,gpt-4o-mini,0,Are participants positive about the interview ...
1,no,The participant's responses indicate a lack of...,"[Bc it's a scheme, They scam people?, The conv...","[018fb4f0-0a82-bae6-39bf-dc6dcf43fc4b, 018fb4f...",018fbf4d-7c57-1615-3ee8-c8b2cfeb1123,2025-05-01 10:56:17.337627+00:00,gpt-4o-mini,0,Are participants positive about the interview ...
2,yes,The participant expresses a positive sentiment...,[No I think l have been able to answer the que...,[018fbfd3-2b96-12e7-db40-a4f809c23c60],018fbfac-da9d-47f8-424c-daf761a75880,2025-05-01 10:56:17.339407+00:00,gpt-4o-mini,0,Are participants positive about the interview ...
3,yes,The participant expresses gratitude and engage...,"[Thank you, You're welcome! If you have any la...","[018fc3b9-5cbb-2819-9fae-78b4059de7d0, 018fc3c...",018fc39f-258a-ea2a-1976-641403a1ce11,2025-05-01 10:56:17.339787+00:00,gpt-4o-mini,0,Are participants positive about the interview ...
4,yes,The participant expresses positive sentiments ...,[I think it might be a government scheme which...,"[018fe2d8-022d-ed9b-c9b9-e2c8e42efe6b, 018fe2e...",018fe2d4-8daa-33c2-3e17-5bfdcf79e54a,2025-05-01 10:56:17.340054+00:00,gpt-4o-mini,0,Are participants positive about the interview ...
5,yes,The participant expresses that the eligibility...,[It makes sense to me in the current financial...,"[018fb920-82e9-b9a5-3aa6-27db881f7e60, 018fb92...",018fb90c-7014-6318-28bb-86a49bf37d97,2025-05-01 10:56:17.340297+00:00,gpt-4o-mini,0,Do participants believe the eligibility requir...
6,yes,The participant expressed a positive view towa...,"[I think it's great, As I said it confirms the...","[018fbf59-9ff7-7b1b-1d96-18e56e49d206, 018fbf5...",018fbf4d-7c57-1615-3ee8-c8b2cfeb1123,2025-05-01 10:56:17.340989+00:00,gpt-4o-mini,0,Do participants believe the eligibility requir...
7,yes,The participant expressed that they believe th...,[I think that it is right that there are eligi...,"[018fbfb2-80e5-bc57-9a43-9531969bf812, 018fbfb...",018fbfac-da9d-47f8-424c-daf761a75880,2025-05-01 10:56:17.341261+00:00,gpt-4o-mini,0,Do participants believe the eligibility requir...
8,yes,The participant expresses a positive view abou...,[It's actually great it is a way to reduce or ...,"[018fc3a8-c5bd-1619-5f8e-16f59771542d, 018fc3a...",018fc39f-258a-ea2a-1976-641403a1ce11,2025-05-01 10:56:17.341506+00:00,gpt-4o-mini,0,Do participants believe the eligibility requir...
9,yes,The participant expressed that the eligibility...,[It makes complete sense as it is the most eff...,[018fe2e5-707d-f0d9-5606-0286b79e3cd2],018fe2d4-8daa-33c2-3e17-5bfdcf79e54a,2025-05-01 10:56:17.341710+00:00,gpt-4o-mini,0,Do participants believe the eligibility requir...
